# Spark Cluster Configuration — Active Parameters

This notebook displays all **explicitly configured** parameters from `spark-defaults.conf`
with their live values, category, and explanation.

Values are read directly from the running SparkSession — including any runtime overrides.


In [1]:
from pyspark.sql import SparkSession

MASTER = 'spark://spark-master:7077'
spark = (SparkSession.builder
    .appName('cluster-config-active')
    .master(MASTER)
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}  |  Master: {spark.sparkContext.master}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 22:03:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2  |  Master: spark://spark-master:7077


In [2]:
# All explicitly configured parameters with live values, category, and explanation
from pyspark.sql import Row

configured_params = [
    # ── CLUSTER ──────────────────────────────────────────────────────────────
    ("spark.master",                                  "Cluster",       "Address of the Spark master node for cluster mode"),
    ("spark.executor.memory",                         "Cluster",       "Heap memory allocated to each executor. 2g = 2 GB"),
    ("spark.driver.memory",                           "Cluster",       "Heap memory allocated to the driver process"),
    ("spark.executor.cores",                          "Cluster",       "CPU cores per executor — controls task parallelism per node"),

    # ── SQL ───────────────────────────────────────────────────────────────────
    ("spark.sql.shuffle.partitions",                  "SQL",           "Number of partitions for shuffle operations (JOIN, GROUP BY). Default=200, reduced to 24 for local cluster"),
    ("spark.sql.adaptive.enabled",                    "SQL / AQE",     "Adaptive Query Execution — Spark re-optimizes the plan at runtime based on actual statistics"),
    ("spark.sql.adaptive.coalescePartitions.enabled", "SQL / AQE",     "AQE: automatically merges small shuffle partitions into larger ones, reducing task count"),
    ("spark.sql.ansi.enabled",                        "SQL",           "ANSI SQL mode. Disabled (false) for Gluten/Velox compatibility — some Velox operators do not support ANSI mode"),
    ("spark.sql.execution.arrow.pyspark.enabled",     "SQL / Arrow",   "Use Apache Arrow for pandas UDFs and toPandas(). Significantly speeds up data transfer between JVM and Python"),

    # ── CATALOGS ─────────────────────────────────────────────────────────────
    ("spark.sql.extensions",                          "Catalog",       "Registers SQL extensions. Iceberg: DDL/DML. Delta: ALTER TABLE, MERGE, VACUUM, etc."),
    ("spark.sql.catalog.local",                       "Catalog",       "Defines the Iceberg catalog named 'local' — used as local.icedb.table_name"),
    ("spark.sql.catalog.local.type",                  "Catalog",       "Iceberg catalog type: hadoop = filesystem-based, no Hive Metastore required"),
    ("spark.sql.catalog.local.warehouse",             "Catalog",       "Root directory for Iceberg tables on the local filesystem"),
    ("spark.sql.catalog.spark_catalog",               "Catalog",       "Replaces the default Spark catalog with the Delta catalog — enables spark.table('delta_table')"),

    # ── SERIALIZATION ─────────────────────────────────────────────────────────
    ("spark.serializer",                              "Serialization", "Kryo serializer is faster and more compact than the default Java serializer. Recommended for production"),

    # ── EVENT LOG / HISTORY ───────────────────────────────────────────────────
    ("spark.eventLog.enabled",                        "History",       "Enables Spark event logging to files — required for the History Server"),
    ("spark.eventLog.dir",                            "History",       "Directory where event log files are written"),
    ("spark.history.fs.logDirectory",                 "History",       "Directory that the History Server reads to display completed applications"),
    ("spark.history.ui.port",                         "History",       "Port for the Spark History Server UI (http://localhost:18080)"),

    # ── SHUFFLE ───────────────────────────────────────────────────────────────
    ("spark.shuffle.sort.bypassMergeThreshold",       "Shuffle",       "Set to 0: disables BypassMergeSortShuffleWriter. Required for Gluten — columnar serializer is incompatible with bypass shuffle writer"),

    # ── JVM OPTIONS ───────────────────────────────────────────────────────────
    ("spark.driver.extraJavaOptions",                 "JVM",           "Suppresses WARN logs from GlutenFallbackReporter to ERROR level — reduces noise from expected fallbacks"),
    ("spark.executor.extraJavaOptions",               "JVM",           "Same Gluten fallback log suppression on the executor side"),
]

rows = []
for key, category, description in configured_params:
    try:
        value = spark.conf.get(key)
    except Exception:
        value = "(not set)"
    rows.append(Row(Parameter=key, Category=category, Value=value, Description=description))

df = spark.createDataFrame(rows)
df.show(n=len(rows), truncate=False)


[Stage 1:===================>                                       (1 + 2) / 3]

+---------------------------------------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [3]:
# Display as a styled HTML table (more readable in JupyterLab)
pdf = df.toPandas()
pdf = pdf[['Category','Parameter','Value','Description']].sort_values(['Category','Parameter'])

try:
    from IPython.display import display, HTML
    display(HTML(pdf.to_html(index=False, border=1,
        justify='left').replace('<th>', '<th style="text-align:left;padding:4px 8px;background:#2c3e50;color:white">').replace('<td>', '<td style="padding:4px 8px;border-bottom:1px solid #ddd">')))
except ImportError:
    print(pdf.to_string(index=False))


Category,Parameter,Value,Description
Catalog,spark.sql.catalog.local,org.apache.iceberg.spark.SparkCatalog,Defines the Iceberg catalog named 'local' — used as local.icedb.table_name
Catalog,spark.sql.catalog.local.type,hadoop,"Iceberg catalog type: hadoop = filesystem-based, no Hive Metastore required"
Catalog,spark.sql.catalog.local.warehouse,/workspace/data/warehouse/iceberg,Root directory for Iceberg tables on the local filesystem
Catalog,spark.sql.catalog.spark_catalog,org.apache.spark.sql.delta.catalog.DeltaCatalog,Replaces the default Spark catalog with the Delta catalog — enables spark.table('delta_table')
Catalog,spark.sql.extensions,"org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,io.delta.sql.DeltaSparkSessionExtension","Registers SQL extensions. Iceberg: DDL/DML. Delta: ALTER TABLE, MERGE, VACUUM, etc."
Cluster,spark.driver.memory,1g,Heap memory allocated to the driver process
Cluster,spark.executor.cores,2,CPU cores per executor — controls task parallelism per node
Cluster,spark.executor.memory,2g,Heap memory allocated to each executor. 2g = 2 GB
Cluster,spark.master,spark://spark-master:7077,Address of the Spark master node for cluster mode
History,spark.eventLog.dir,/opt/spark/logs,Directory where event log files are written


In [4]:
# Summary by category
print("Parameters by category:")
print()
for cat in sorted(pdf['Category'].unique()):
    count = len(pdf[pdf['Category']==cat])
    print(f"  {cat:<20} {count} parameter(s)")
print()
print(f"Total explicitly configured parameters: {len(pdf)}")


Parameters by category:

  Catalog              5 parameter(s)
  Cluster              4 parameter(s)
  History              4 parameter(s)
  JVM                  2 parameter(s)
  SQL                  2 parameter(s)
  SQL / AQE            2 parameter(s)
  SQL / Arrow          1 parameter(s)
  Serialization        1 parameter(s)
  Shuffle              1 parameter(s)

Total explicitly configured parameters: 22
